# Hy3 API - 错误处理与重试示例

本示例演示 Hy3 API 的错误处理与重试机制，包括超时、限流、网络错误的重试与指数退避策略。

In [ ]:
from openai import OpenAI, APIError, APITimeoutError, APIConnectionError, RateLimitError
import os
import time
import random
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(
    api_key=os.getenv("API_KEY", "EMPTY"),
    base_url=os.getenv("BASE_URL", "http://127.0.0.1:8000/v1"),
    timeout=5.0,
)



## 错误类型

| 错误类型 | 说明 | 是否可重试 |
|:---|:---|:---|
| `APITimeoutError` | 请求超时 | 是 |
| `APIConnectionError` | 网络连接失败 | 是 |
| `RateLimitError` | 请求被限流 | 是 |
| `APIError` | 服务端错误 | 视情况 |
| `AuthenticationError` | 认证失败 | 否 |

In [ ]:
def retry_with_backoff(func, max_retries=3, base_delay=1.0, max_delay=10.0):
    def wrapper(*args, **kwargs):
        retries = 0
        delay = base_delay

        while retries < max_retries:
            try:
                return func(*args, **kwargs)
            except (APITimeoutError, APIConnectionError) as e:
                retries += 1
                if retries >= max_retries:
                    raise
                jitter = random.uniform(0, delay * 0.1)
                wait_time = delay + jitter
                print(f"  网络错误: {type(e).__name__} - 第 {retries}/{max_retries} 次重试，等待 {wait_time:.2f}s")
                time.sleep(wait_time)
                delay = min(delay * 2, max_delay)
            except RateLimitError as e:
                retries += 1
                if retries >= max_retries:
                    raise
                reset_after = getattr(e, 'reset_after', 1)
                wait_time = reset_after + random.uniform(0, 1)
                print(f"  限流错误: {type(e).__name__} - 第 {retries}/{max_retries} 次重试，等待 {wait_time:.2f}s")
                time.sleep(wait_time)
            except APIError as e:
                print(f"  API错误: {type(e).__name__} - {e}")
                raise

    return wrapper

## 场景 1：超时处理

In [ ]:
slow_client = OpenAI(
    api_key=os.getenv("API_KEY", "EMPTY"),
    base_url=os.getenv("BASE_URL", "http://127.0.0.1:8000/v1"),
    timeout=1.0,  # 故意设置很短以触发超时
)

try:
    response = slow_client.chat.completions.create(
        model="hy3",
        messages=[{"role": "user", "content": "请写一篇关于人工智能的长文"}],
        temperature=0.9,
        top_p=1.0,
    )
    print("【成功】响应正常返回")
except APITimeoutError as e:
    print(f"【超时】请求在1秒内未完成")
    print(f"  错误: {type(e).__name__}")

## 场景 2：重试装饰器测试

In [ ]:
@retry_with_backoff
def chat_with_retry(messages):
    return client.chat.completions.create(
        model="hy3",
        messages=messages,
        temperature=0.9,
        top_p=1.0,
    )

try:
    response = chat_with_retry([{"role": "user", "content": "你好"}])
    
    print("【成功响应解析】")
    print(f"  id: {response.id}")
    print(f"  model: {response.model}")
    print(f"  finish_reason: {response.choices[0].finish_reason}")
    print(f"  content: {response.choices[0].message.content}")

except APITimeoutError as e:
    print(f"【超时错误】请求超时，已达到最大重试次数")

except APIConnectionError as e:
    print(f"【网络错误】连接失败，已达到最大重试次数")

except RateLimitError as e:
    print(f"【限流错误】请求被限流，已达到最大重试次数")

except APIError as e:
    print(f"【API错误】服务端错误")

## 场景 3：限流处理测试

In [ ]:
@retry_with_backoff
def limited_request():
    return client.chat.completions.create(
        model="hy3",
        messages=[{"role": "user", "content": "测试"}],
    )

print("【模拟高频请求】")
for i in range(5):
    try:
        print(f"  请求 #{i+1}...")
        response = limited_request()
        print(f"    成功: {len(response.choices[0].message.content)} 字符")
    except Exception as e:
        print(f"    失败: {type(e).__name__}")
        break

## 最佳实践

1. **设置合理超时**：建议设置 5-10 秒超时
2. **指数退避**：避免重试风暴
3. **Jitter**：防止多个客户端同时重试
4. **限流感知**：使用 `reset_after` 字段
5. **最大重试次数**：设置合理上限（3-5 次）
6. **区分错误类型**：认证错误不应重试